# RxGuide AI — SQL Business Analysis

This notebook uses SQL to analyse HCP value, field engagement, product growth, territory performance, and representative performance.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


# Find the project root automatically
current_directory = Path.cwd().resolve()
PROJECT_ROOT = None

for directory in [current_directory, *current_directory.parents]:
    if (directory / "database" / "rxguide_analytics.db").exists():
        PROJECT_ROOT = directory
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find database/rxguide_analytics.db"
    )

DATABASE_PATH = (
    PROJECT_ROOT
    / "database"
    / "rxguide_analytics.db"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

connection = sqlite3.connect(DATABASE_PATH)

print("Connected to:", DATABASE_PATH)

Connected to: D:\rxguide\rxguide-hcp-analytics\database\rxguide_analytics.db


In [2]:
tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

display(tables)

,table_name
0,call_activity
1,hcps
2,ic_quotas
3,monthly_prescriptions
4,products
5,sales_reps


In [3]:
table_names = tables["table_name"].tolist()

row_counts = []

for table_name in table_names:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {table_name}",
        connection
    ).iloc[0]["row_count"]

    row_counts.append(
        {
            "table_name": table_name,
            "row_count": count,
        }
    )

row_count_report = pd.DataFrame(row_counts)

display(row_count_report)

,table_name,row_count
0,call_activity,13705
1,hcps,500
2,ic_quotas,200
3,monthly_prescriptions,14452
4,products,5
5,sales_reps,50


## 1. High-Value but Under-Covered HCPs

This analysis identifies HCPs who belong to the highest quartile of medication volume but the lowest quartile of field-call coverage.

In [6]:
high_value_under_covered_query = """
WITH analysis_window AS (
    SELECT
        DATE(
            MAX(rx_month),
            '+1 month'
        ) AS end_date_exclusive
    FROM monthly_prescriptions
),

call_summary AS (
    SELECT
        c.hcp_id,
        COUNT(*) AS total_calls,
        MAX(c.call_date) AS last_call_date
    FROM call_activity AS c

    CROSS JOIN analysis_window AS w

    WHERE c.call_date < w.end_date_exclusive

    GROUP BY c.hcp_id
),
prescription_summary AS (
    SELECT
        hcp_id,
        SUM(rx_count) AS total_prescriptions,
        SUM(units_dispensed) AS total_units_dispensed,
        COUNT(DISTINCT product_id) AS product_breadth
    FROM monthly_prescriptions
    GROUP BY hcp_id
),

hcp_metrics AS (
    SELECT
        h.hcp_id,
        h.hcp_name,
        h.specialty,
        h.segment,
        h.territory,
        h.region,

        COALESCE(c.total_calls, 0) AS total_calls,
        c.last_call_date,

        COALESCE(
            p.total_prescriptions,
            0
        ) AS total_prescriptions,

        COALESCE(
            p.total_units_dispensed,
            0
        ) AS total_units_dispensed,

        COALESCE(
            p.product_breadth,
            0
        ) AS product_breadth

    FROM hcps AS h

    LEFT JOIN call_summary AS c
        ON h.hcp_id = c.hcp_id

    LEFT JOIN prescription_summary AS p
        ON h.hcp_id = p.hcp_id
),

ranked_hcps AS (
    SELECT
        *,

        PERCENT_RANK() OVER (
            ORDER BY total_units_dispensed
        ) AS value_percentile,

        PERCENT_RANK() OVER (
            ORDER BY total_calls DESC
        ) AS undercoverage_percentile

    FROM hcp_metrics

    WHERE total_units_dispensed > 0
),

scored_hcps AS (
    SELECT
        *,

        ROUND(
            (
                0.65 * value_percentile
                +
                0.35 * undercoverage_percentile
            ) * 100,
            2
        ) AS opportunity_score

    FROM ranked_hcps
)

SELECT
    hcp_id,
    hcp_name,
    specialty,
    segment,
    territory,
    region,
    total_calls,
    total_prescriptions,
    total_units_dispensed,
    product_breadth,
    last_call_date,

    ROUND(
        total_prescriptions * 1.0
        / NULLIF(total_calls, 0),
        2
    ) AS prescriptions_per_call,

    ROUND(
        value_percentile * 100,
        2
    ) AS value_percentile,

    ROUND(
        undercoverage_percentile * 100,
        2
    ) AS undercoverage_percentile,

    opportunity_score,

    CASE
        WHEN total_calls = 0
            THEN 'No Field Coverage'
        WHEN undercoverage_percentile >= 0.75
            THEN 'Low Coverage'
        ELSE 'Regular Coverage'
    END AS coverage_status

FROM scored_hcps

ORDER BY opportunity_score DESC

LIMIT 20;
"""

hcp_opportunity_ranking = pd.read_sql_query(
    high_value_under_covered_query,
  
    connection
)

display(hcp_opportunity_ranking)

,hcp_id,hcp_name,specialty,segment,territory,region,total_calls,total_prescriptions,total_units_dispensed,product_breadth,last_call_date,prescriptions_per_call,value_percentile,undercoverage_percentile,opportunity_score,coverage_status
0,H0459,Dr. HCP_0459,Pulmonologist,A,North-1,North,17,184,7937,5,2026-03-27,10.82,96.07,57.35,82.52,Regular Coverage
1,H0038,Dr. HCP_0038,Neurologist,A,North-2,North,15,143,5904,5,2026-03-24,9.53,89.23,63.56,80.25,Regular Coverage
2,H0016,Dr. HCP_0016,General Physician,A,North-1,North,11,119,5083,5,2026-03-19,10.82,82.19,75.16,79.73,Low Coverage
3,H0291,Dr. HCP_0291,Oncologist,A,South-2,South,22,200,8556,5,2026-03-20,9.09,97.72,45.34,79.39,Regular Coverage
4,H0333,Dr. HCP_0333,Cardiologist,A,South-2,South,22,200,8321,5,2026-03-24,9.09,96.89,45.34,78.85,Regular Coverage
5,H0300,Dr. HCP_0300,General Physician,A,North-2,North,19,169,6909,5,2026-03-26,8.89,92.96,51.97,78.61,Regular Coverage
6,H0373,Dr. HCP_0373,Oncologist,A,North-2,North,24,191,8475,5,2026-03-27,7.96,97.31,42.24,78.03,Regular Coverage
7,H0069,Dr. HCP_0069,Endocrinologist,A,South-2,South,26,224,9832,5,2026-03-31,8.62,99.17,38.72,78.01,Regular Coverage
8,H0245,Dr. HCP_0245,Cardiologist,A,South-2,South,28,287,11848,5,2026-03-26,10.25,99.79,36.23,77.55,Regular Coverage
9,H0099,Dr. HCP_0099,Endocrinologist,A,South-2,South,21,170,7095,5,2026-03-30,8.10,93.79,47.20,77.48,Regular Coverage


In [7]:
hcp_opportunity_ranking.to_csv(
    OUTPUT_DIR / "sql_hcp_opportunity_ranking.csv",
    index=False
)

print(
    "HCP opportunity ranking exported successfully."
)

HCP opportunity ranking exported successfully.


## Sales Representative Efficiency

This analysis compares sales representatives using target attainment, credited performance units, HCP reach, call volume, and units generated per call.

In [8]:
rep_efficiency_query = """
WITH analysis_window AS (
    SELECT
        MIN(quarter_start) AS start_date,
        MAX(quarter_end) AS end_date
    FROM ic_quotas
),

call_summary AS (
    SELECT
        c.rep_id,
        COUNT(*) AS total_calls,
        COUNT(DISTINCT c.hcp_id) AS unique_hcps_reached,
        COUNT(DISTINCT c.product_id) AS products_discussed,
        SUM(c.samples_given) AS total_samples_given,
        ROUND(AVG(c.duration_min), 2) AS average_call_duration
    FROM call_activity AS c

    CROSS JOIN analysis_window AS w

    WHERE c.call_date BETWEEN w.start_date AND w.end_date

    GROUP BY c.rep_id
),

quota_summary AS (
    SELECT
        rep_id,
        COUNT(DISTINCT quarter) AS quarters_measured,
        SUM(quota_units) AS total_quota_units,
        SUM(actual_units) AS total_actual_units,
        SUM(payout_inr) AS total_incentive_payout
    FROM ic_quotas
    GROUP BY rep_id
),

rep_metrics AS (
    SELECT
        r.rep_id,
        r.rep_name,
        r.territory,
        r.region,
        r.manager,
        r.tenure_months,

        COALESCE(c.total_calls, 0) AS total_calls,
        COALESCE(c.unique_hcps_reached, 0)
            AS unique_hcps_reached,
        COALESCE(c.products_discussed, 0)
            AS products_discussed,
        COALESCE(c.total_samples_given, 0)
            AS total_samples_given,
        c.average_call_duration,

        q.quarters_measured,
        q.total_quota_units,
        q.total_actual_units,
        q.total_incentive_payout,

        ROUND(
            q.total_actual_units * 100.0
            / NULLIF(q.total_quota_units, 0),
            2
        ) AS weighted_attainment_pct,

        ROUND(
            q.total_actual_units * 1.0
            / NULLIF(c.total_calls, 0),
            2
        ) AS actual_units_per_call,

        ROUND(
            c.total_calls * 1.0
            / NULLIF(c.unique_hcps_reached, 0),
            2
        ) AS calls_per_hcp

    FROM sales_reps AS r

    LEFT JOIN call_summary AS c
        ON r.rep_id = c.rep_id

    LEFT JOIN quota_summary AS q
        ON r.rep_id = q.rep_id
)

SELECT
    *,

    DENSE_RANK() OVER (
        ORDER BY weighted_attainment_pct DESC
    ) AS attainment_rank,

    DENSE_RANK() OVER (
        ORDER BY actual_units_per_call DESC
    ) AS efficiency_rank,

    CASE
        WHEN weighted_attainment_pct >= 100
            THEN 'Exceeded Target'
        WHEN weighted_attainment_pct >= 90
            THEN 'Near Target'
        ELSE 'Below Target'
    END AS performance_status

FROM rep_metrics

ORDER BY
    weighted_attainment_pct DESC,
    actual_units_per_call DESC;
"""

rep_efficiency = pd.read_sql_query(
    rep_efficiency_query,
    connection
)

display(rep_efficiency.head(15))

,rep_id,rep_name,territory,region,manager,tenure_months,total_calls,unique_hcps_reached,products_discussed,total_samples_given,average_call_duration,quarters_measured,total_quota_units,total_actual_units,total_incentive_payout,weighted_attainment_pct,actual_units_per_call,calls_per_hcp,attainment_rank,efficiency_rank,performance_status
0,R016,Rep_016,North-2,North,Sneha Patel,91,201,57,5,811,19.44,4,6339,7298,345123,115.13,36.31,3.53,1,2,Exceeded Target
1,R010,Rep_010,North-2,North,Sneha Patel,79,197,54,5,787,19.38,4,4798,5315,322227,110.78,26.98,3.65,2,19,Exceeded Target
2,R027,Rep_027,West-1,West,Vikram Singh,79,244,52,5,1014,18.75,4,5686,6292,325365,110.66,25.79,4.69,3,20,Exceeded Target
3,R046,Rep_046,South-2,South,Anil Mehta,81,283,48,5,1076,19.47,4,5208,5746,326309,110.33,20.30,5.90,4,29,Exceeded Target
4,R040,Rep_040,North-1,North,Rajesh Kumar,72,322,33,5,1309,19.50,4,5029,5545,322172,110.26,17.22,9.76,5,42,Exceeded Target
5,R009,Rep_009,South-1,South,Priya Sharma,44,340,44,5,1348,20.17,4,5300,5841,334652,110.21,17.18,7.73,6,43,Exceeded Target
6,R001,Rep_001,North-2,North,Rajesh Kumar,63,215,55,5,913,20.68,4,5778,6326,333095,109.48,29.42,3.91,7,14,Exceeded Target
7,R024,Rep_024,South-2,South,Sneha Patel,92,226,46,5,867,18.92,4,7269,7919,311248,108.94,35.04,4.91,8,4,Exceeded Target
8,R030,Rep_030,South-1,South,Sneha Patel,39,374,43,5,1587,19.65,4,6471,6913,320295,106.83,18.48,8.70,9,38,Exceeded Target
9,R043,Rep_043,Central-1,Central,Priya Sharma,94,296,46,5,1202,20.29,4,6539,6961,318896,106.45,23.52,6.43,10,23,Exceeded Target


In [9]:
rep_efficiency.to_csv(
    OUTPUT_DIR / "sql_representative_efficiency.csv",
    index=False
)

print("Representative efficiency result exported successfully.")

Representative efficiency result exported successfully.


## Product Month-over-Month Growth

This analysis compares each product's monthly medication volume with its previous month's performance.

In [10]:
product_growth_query = """
WITH product_monthly AS (
    SELECT
        mp.product_id,
        p.product_name,
        p.therapy_area,
        mp.rx_month,

        SUM(mp.rx_count) AS total_prescriptions,
        SUM(mp.units_dispensed) AS total_units_dispensed,
        COUNT(DISTINCT mp.hcp_id) AS prescribing_hcps

    FROM monthly_prescriptions AS mp

    INNER JOIN products AS p
        ON mp.product_id = p.product_id

    GROUP BY
        mp.product_id,
        p.product_name,
        p.therapy_area,
        mp.rx_month
),

previous_month_values AS (
    SELECT
        *,

        LAG(total_units_dispensed) OVER (
            PARTITION BY product_id
            ORDER BY rx_month
        ) AS previous_month_units,

        LAG(total_prescriptions) OVER (
            PARTITION BY product_id
            ORDER BY rx_month
        ) AS previous_month_prescriptions

    FROM product_monthly
)

SELECT
    product_id,
    product_name,
    therapy_area,
    rx_month,
    total_prescriptions,
    total_units_dispensed,
    prescribing_hcps,
    previous_month_units,

    total_units_dispensed
        - previous_month_units
        AS unit_change,

    ROUND(
        (
            total_units_dispensed
            - previous_month_units
        ) * 100.0
        / NULLIF(previous_month_units, 0),
        2
    ) AS unit_growth_pct,

    ROUND(
        (
            total_prescriptions
            - previous_month_prescriptions
        ) * 100.0
        / NULLIF(previous_month_prescriptions, 0),
        2
    ) AS prescription_growth_pct

FROM previous_month_values

ORDER BY
    product_name,
    rx_month;
"""

product_monthly_growth = pd.read_sql_query(
    product_growth_query,
    connection
)

display(product_monthly_growth.head(15))

,product_id,product_name,therapy_area,rx_month,total_prescriptions,total_units_dispensed,prescribing_hcps,previous_month_units,unit_change,unit_growth_pct,prescription_growth_pct
0,P001,Cardiozen,Cardiology,2025-06-01,740,29707,289,NaN,NaN,NaN,NaN
1,P001,Cardiozen,Cardiology,2025-07-01,695,27463,289,"29,707.00","-2,244.00",-7.55,-6.08
2,P001,Cardiozen,Cardiology,2025-08-01,714,28783,288,"27,463.00","1,320.00",4.81,2.73
3,P001,Cardiozen,Cardiology,2025-09-01,698,28567,303,"28,783.00",-216.00,-0.75,-2.24
4,P001,Cardiozen,Cardiology,2025-10-01,720,29042,297,"28,567.00",475.00,1.66,3.15
5,P001,Cardiozen,Cardiology,2025-11-01,744,29744,311,"29,042.00",702.00,2.42,3.33
6,P001,Cardiozen,Cardiology,2025-12-01,688,27132,303,"29,744.00","-2,612.00",-8.78,-7.53
7,P001,Cardiozen,Cardiology,2026-01-01,735,30814,301,"27,132.00","3,682.00",13.57,6.83
8,P001,Cardiozen,Cardiology,2026-02-01,747,29289,300,"30,814.00","-1,525.00",-4.95,1.63
9,P001,Cardiozen,Cardiology,2026-03-01,715,29296,294,"29,289.00",7.00,0.02,-4.28


In [11]:
latest_product_growth = (
    product_monthly_growth[
        product_monthly_growth["rx_month"]
        == product_monthly_growth["rx_month"].max()
    ]
    .sort_values(
        "unit_growth_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(latest_product_growth)


,product_id,product_name,therapy_area,rx_month,total_prescriptions,total_units_dispensed,prescribing_hcps,previous_month_units,unit_change,unit_growth_pct,prescription_growth_pct
0,P005,Neurofine,Neurology,2026-03-01,687,28195,283,"27,789.00",406.00,1.46,-1.58
1,P004,Oncoshield,Oncology,2026-03-01,697,27712,286,"27,606.00",106.00,0.38,-1.13
2,P001,Cardiozen,Cardiology,2026-03-01,715,29296,294,"29,289.00",7.00,0.02,-4.28
3,P002,Diabolite,Diabetes,2026-03-01,720,28836,301,"30,743.00","-1,907.00",-6.20,-2.31
4,P003,Respira-X,Respiratory,2026-03-01,640,24752,268,"30,240.00","-5,488.00",-18.15,-12.09


In [12]:
product_monthly_growth.to_csv(
    OUTPUT_DIR / "sql_product_monthly_growth.csv",
    index=False
)

print("Product growth analysis exported successfully.")

Product growth analysis exported successfully.


## Territory Performance and Coverage

This analysis compares territory output, call efficiency, HCP coverage, and sales-representative availability.

In [13]:
territory_performance_query = """
WITH analysis_window AS (
    SELECT
        MIN(rx_month) AS start_date,
        DATE(MAX(rx_month), '+1 month')
            AS end_date_exclusive
    FROM monthly_prescriptions
),

territory_master AS (
    SELECT
        territory,
        region,
        COUNT(DISTINCT hcp_id) AS total_hcps
    FROM hcps
    GROUP BY territory, region
),

rep_summary AS (
    SELECT
        territory,
        region,
        COUNT(DISTINCT rep_id) AS assigned_reps
    FROM sales_reps
    GROUP BY territory, region
),

call_summary AS (
    SELECT
        h.territory,
        h.region,
        COUNT(*) AS total_calls,
        COUNT(DISTINCT c.hcp_id) AS covered_hcps,
        SUM(c.samples_given) AS total_samples
    FROM call_activity AS c

    INNER JOIN hcps AS h
        ON c.hcp_id = h.hcp_id

    CROSS JOIN analysis_window AS w

    WHERE c.call_date >= w.start_date
      AND c.call_date < w.end_date_exclusive

    GROUP BY h.territory, h.region
),

prescription_summary AS (
    SELECT
        h.territory,
        h.region,
        SUM(mp.rx_count) AS total_prescriptions,
        SUM(mp.units_dispensed) AS total_units_dispensed,
        COUNT(DISTINCT mp.hcp_id) AS prescribing_hcps
    FROM monthly_prescriptions AS mp

    INNER JOIN hcps AS h
        ON mp.hcp_id = h.hcp_id

    GROUP BY h.territory, h.region
),

territory_metrics AS (
    SELECT
        t.territory,
        t.region,
        t.total_hcps,

        COALESCE(r.assigned_reps, 0)
            AS assigned_reps,

        COALESCE(c.total_calls, 0)
            AS total_calls,

        COALESCE(c.covered_hcps, 0)
            AS covered_hcps,

        COALESCE(c.total_samples, 0)
            AS total_samples,

        COALESCE(p.total_prescriptions, 0)
            AS total_prescriptions,

        COALESCE(p.total_units_dispensed, 0)
            AS total_units_dispensed,

        COALESCE(p.prescribing_hcps, 0)
            AS prescribing_hcps

    FROM territory_master AS t

    LEFT JOIN rep_summary AS r
        ON t.territory = r.territory
       AND t.region = r.region

    LEFT JOIN call_summary AS c
        ON t.territory = c.territory
       AND t.region = c.region

    LEFT JOIN prescription_summary AS p
        ON t.territory = p.territory
       AND t.region = p.region
)

SELECT
    *,

    ROUND(
        covered_hcps * 100.0
        / NULLIF(total_hcps, 0),
        2
    ) AS hcp_coverage_pct,

    ROUND(
        total_units_dispensed * 1.0
        / NULLIF(total_calls, 0),
        2
    ) AS units_per_call,

    ROUND(
        total_prescriptions * 1.0
        / NULLIF(total_calls, 0),
        2
    ) AS prescriptions_per_call,

    ROUND(
        total_units_dispensed * 1.0
        / NULLIF(prescribing_hcps, 0),
        2
    ) AS units_per_prescribing_hcp,

    CASE
        WHEN assigned_reps = 0
             AND prescribing_hcps > 0
            THEN 'Uncovered Opportunity'

        WHEN covered_hcps * 100.0
             / NULLIF(total_hcps, 0) < 60
            THEN 'Coverage Gap'

        ELSE 'Active Coverage'
    END AS territory_status

FROM territory_metrics

ORDER BY
    total_units_dispensed DESC;
"""

sql_territory_performance = pd.read_sql_query(
    territory_performance_query,
    connection
)

display(sql_territory_performance)

,territory,region,total_hcps,assigned_reps,total_calls,covered_hcps,total_samples,total_prescriptions,total_units_dispensed,prescribing_hcps,hcp_coverage_pct,units_per_call,prescriptions_per_call,units_per_prescribing_hcp,territory_status
0,South-2,South,48,10,2363,48,9418,8037,320419,48,100.00,135.60,3.40,"6,675.40",Active Coverage
1,North-2,North,57,10,2141,57,8603,7197,290466,57,100.00,135.67,3.36,"5,095.89",Active Coverage
2,Central-1,Central,46,7,1631,46,6491,4335,171254,46,100.00,105.00,2.66,"3,722.91",Active Coverage
3,North-1,North,34,5,1203,34,4772,3745,151775,34,100.00,126.16,3.11,"4,463.97",Active Coverage
4,South-1,South,44,5,1256,44,4994,3606,142449,44,100.00,113.41,2.87,"3,237.48",Active Coverage
5,West-2,West,59,4,780,59,2996,2557,101922,59,100.00,130.67,3.28,"1,727.49",Active Coverage
6,West-1,West,54,3,803,54,3176,2381,96364,54,100.00,120.00,2.97,"1,784.52",Active Coverage
7,East-2,East,53,3,652,53,2611,2029,82445,53,100.00,126.45,3.11,"1,555.57",Active Coverage
8,Central-2,Central,49,3,556,49,2241,1841,75125,49,100.00,135.12,3.31,"1,533.16",Active Coverage
9,East-1,East,56,0,0,0,0,206,8294,40,0.00,NaN,NaN,207.35,Uncovered Opportunity


In [14]:
sql_territory_performance.to_csv(
    OUTPUT_DIR / "sql_territory_performance.csv",
    index=False
)

print("Territory performance analysis exported successfully.")

Territory performance analysis exported successfully.


## HCP Feature View for Segmentation

This SQL view combines prescription behaviour, field engagement, product breadth, recent growth, call efficiency, and engagement recency into one HCP-level analytical table.

In [15]:
feature_view_sql = """
DROP VIEW IF EXISTS vw_hcp_features;

CREATE VIEW vw_hcp_features AS

WITH analysis_window AS (
    SELECT
        MIN(rx_month) AS start_date,
        MAX(rx_month) AS latest_rx_month,

        DATE(
            MAX(rx_month),
            '+1 month'
        ) AS end_date_exclusive,

        DATE(
            MAX(rx_month),
            '+1 month',
            '-1 day'
        ) AS reference_date,

        (
            (
                CAST(
                    STRFTIME(
                        '%Y',
                        DATE(MAX(rx_month), '+1 month')
                    ) AS INTEGER
                )
                -
                CAST(
                    STRFTIME('%Y', MIN(rx_month))
                    AS INTEGER
                )
            ) * 12
            +
            (
                CAST(
                    STRFTIME(
                        '%m',
                        DATE(MAX(rx_month), '+1 month')
                    ) AS INTEGER
                )
                -
                CAST(
                    STRFTIME('%m', MIN(rx_month))
                    AS INTEGER
                )
            )
        ) AS months_in_window

    FROM monthly_prescriptions
),

call_summary AS (
    SELECT
        c.hcp_id,
        COUNT(*) AS total_calls,
        SUM(c.samples_given) AS total_samples_received,
        ROUND(AVG(c.duration_min), 2)
            AS average_call_duration,
        COUNT(DISTINCT c.product_id)
            AS called_product_breadth,
        MAX(c.call_date) AS last_call_date

    FROM call_activity AS c

    CROSS JOIN analysis_window AS w

    WHERE c.call_date >= w.start_date
      AND c.call_date < w.end_date_exclusive

    GROUP BY c.hcp_id
),

hcp_monthly_prescriptions AS (
    SELECT
        hcp_id,
        rx_month,
        SUM(rx_count) AS monthly_prescriptions,
        SUM(units_dispensed) AS monthly_units

    FROM monthly_prescriptions

    GROUP BY
        hcp_id,
        rx_month
),

prescription_summary AS (
    SELECT
        hcp_id,
        SUM(monthly_prescriptions)
            AS total_prescriptions,
        SUM(monthly_units)
            AS total_units_dispensed,
        COUNT(DISTINCT rx_month)
            AS active_months

    FROM hcp_monthly_prescriptions

    GROUP BY hcp_id
),

product_summary AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT product_id)
            AS product_breadth

    FROM monthly_prescriptions

    GROUP BY hcp_id
),

trend_summary AS (
    SELECT
        hm.hcp_id,

        SUM(
            CASE
                WHEN hm.rx_month >= DATE(
                    w.latest_rx_month,
                    '-2 months'
                )
                THEN hm.monthly_units
                ELSE 0
            END
        ) AS recent_3m_units,

        SUM(
            CASE
                WHEN hm.rx_month >= DATE(
                    w.latest_rx_month,
                    '-5 months'
                )
                 AND hm.rx_month < DATE(
                    w.latest_rx_month,
                    '-2 months'
                )
                THEN hm.monthly_units
                ELSE 0
            END
        ) AS previous_3m_units

    FROM hcp_monthly_prescriptions AS hm

    CROSS JOIN analysis_window AS w

    GROUP BY hm.hcp_id
)

SELECT
    h.hcp_id,
    h.hcp_name,
    h.specialty,
    h.segment,
    h.territory,
    h.region,

    COALESCE(p.total_prescriptions, 0)
        AS total_prescriptions,

    COALESCE(p.total_units_dispensed, 0)
        AS total_units_dispensed,

    ROUND(
        COALESCE(p.total_prescriptions, 0) * 1.0
        / w.months_in_window,
        2
    ) AS average_monthly_prescriptions,

    COALESCE(p.active_months, 0)
        AS active_months,

    COALESCE(ps.product_breadth, 0)
        AS product_breadth,

    COALESCE(c.total_calls, 0)
        AS total_calls,

    COALESCE(c.total_samples_received, 0)
        AS total_samples_received,

    COALESCE(c.average_call_duration, 0)
        AS average_call_duration,

    COALESCE(c.called_product_breadth, 0)
        AS called_product_breadth,

    c.last_call_date,

    CASE
        WHEN COALESCE(c.total_calls, 0) > 0
        THEN ROUND(
            p.total_prescriptions * 1.0
            / c.total_calls,
            2
        )
        ELSE NULL
    END AS prescriptions_per_call,

    CASE
        WHEN c.last_call_date IS NULL
        THEN CAST(
            JULIANDAY(w.reference_date)
            - JULIANDAY(w.start_date)
            + 1
            AS INTEGER
        )

        ELSE CAST(
            JULIANDAY(w.reference_date)
            - JULIANDAY(c.last_call_date)
            AS INTEGER
        )
    END AS days_since_last_call,

    CASE
        WHEN COALESCE(c.total_calls, 0) = 0
        THEN 1
        ELSE 0
    END AS no_call_flag,

    COALESCE(t.recent_3m_units, 0)
        AS recent_3m_units,

    COALESCE(t.previous_3m_units, 0)
        AS previous_3m_units,

    CASE
        WHEN t.previous_3m_units > 0
        THEN ROUND(
            (
                t.recent_3m_units
                - t.previous_3m_units
            ) * 100.0
            / t.previous_3m_units,
            2
        )
        ELSE NULL
    END AS recent_growth_pct

FROM hcps AS h

CROSS JOIN analysis_window AS w

LEFT JOIN call_summary AS c
    ON h.hcp_id = c.hcp_id

LEFT JOIN prescription_summary AS p
    ON h.hcp_id = p.hcp_id

LEFT JOIN product_summary AS ps
    ON h.hcp_id = ps.hcp_id

LEFT JOIN trend_summary AS t
    ON h.hcp_id = t.hcp_id;
"""

with connection:
    connection.executescript(feature_view_sql)

print("HCP feature view created successfully.")

HCP feature view created successfully.


In [16]:
hcp_features = pd.read_sql_query(
    """
    SELECT *
    FROM vw_hcp_features
    ORDER BY hcp_id;
    """,
    connection
)

print("Feature rows:", len(hcp_features))
print("Unique HCPs:", hcp_features["hcp_id"].nunique())
print("HCPs with no calls:", hcp_features["no_call_flag"].sum())

display(hcp_features.head(10))

Feature rows: 500
Unique HCPs: 500
HCPs with no calls: 56


,hcp_id,hcp_name,specialty,segment,territory,region,total_prescriptions,total_units_dispensed,average_monthly_prescriptions,active_months,product_breadth,total_calls,total_samples_received,average_call_duration,called_product_breadth,last_call_date,prescriptions_per_call,days_since_last_call,no_call_flag,recent_3m_units,previous_3m_units,recent_growth_pct
0,H0001,Dr. HCP_0001,Endocrinologist,B,North-1,North,91,3711,9.10,10,5,27,99,21.56,5,2026-03-30,3.37,1,0,958,1067,-10.22
1,H0002,Dr. HCP_0002,Pulmonologist,C,North-1,North,61,2439,6.10,10,5,55,238,20.78,5,2026-03-28,1.11,3,0,864,846,2.13
2,H0003,Dr. HCP_0003,Endocrinologist,C,Central-1,Central,89,3217,8.90,10,5,40,155,20.35,5,2026-03-22,2.23,9,0,874,982,-11.00
3,H0004,Dr. HCP_0004,Endocrinologist,B,West-2,West,44,1809,4.40,10,5,14,58,21.07,5,2026-02-23,3.14,36,0,641,299,114.38
4,H0005,Dr. HCP_0005,Neurologist,A,Central-2,Central,29,1179,2.90,10,4,6,26,22.00,3,2026-03-04,4.83,27,0,411,368,11.68
5,H0006,Dr. HCP_0006,Oncologist,A,South-2,South,206,7914,20.60,10,5,26,118,17.46,5,2026-03-30,7.92,1,0,2671,2706,-1.29
6,H0007,Dr. HCP_0007,Cardiologist,A,South-2,South,217,8517,21.70,10,5,25,104,17.92,5,2026-03-23,8.68,8,0,2971,2523,17.76
7,H0008,Dr. HCP_0008,Pulmonologist,C,West-1,West,23,964,2.30,10,3,17,82,20.76,5,2026-03-17,1.35,14,0,236,363,-34.99
8,H0009,Dr. HCP_0009,Oncologist,B,West-1,West,48,2111,4.80,9,4,10,36,21.30,4,2026-03-29,4.80,2,0,626,557,12.39
9,H0010,Dr. HCP_0010,Cardiologist,C,North-2,North,68,2853,6.80,10,5,56,254,18.80,5,2026-03-25,1.21,6,0,786,941,-16.47


In [17]:
missing_feature_values = (
    hcp_features
    .isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

display(missing_feature_values)

,missing_values
last_call_date,56
prescriptions_per_call,56
recent_growth_pct,23


In [18]:
PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

hcp_features.to_csv(
    PROCESSED_DATA_DIR / "hcp_ml_features.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("HCP ML feature table exported successfully.")

HCP ML feature table exported successfully.
